In [34]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

from xgboost import XGBRegressor

In [48]:
df = pd.read_csv("train.csv")

print(df.shape)
df.head()

(8523, 12)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [49]:
# Item_Weight missing values
df['Item_Weight'].fillna(df['Item_Weight'].mean(), inplace=True)

# Outlet_Size missing values
df['Outlet_Size'].fillna(df['Outlet_Size'].mode()[0], inplace=True)

C:\Users\HARSH GOSALIA\AppData\Local\Temp\ipykernel_13572\2659572703.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Item_Weight'].fillna(df['Item_Weight'].mean(), inplace=True)
C:\Users\HARSH GOSALIA\AppData\Local\Temp\ipykernel_13572\2659572703.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or S

0       Medium
1       Medium
2       Medium
3       Medium
4         High
         ...  
8518      High
8519    Medium
8520     Small
8521    Medium
8522     Small
Name: Outlet_Size, Length: 8523, dtype: str

In [50]:
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'LF':'Low Fat',
    'low fat':'Low Fat',
    'reg':'Regular'
})

In [51]:
df['Item_Type_Combined'] = df['Item_Identifier'].apply(lambda x: x[:2])

df['Item_Type_Combined'] = df['Item_Type_Combined'].map({
    'FD':'Food',
    'NC':'Non-Consumable',
    'DR':'Drinks'
})

In [52]:
df['Outlet_Type'].value_counts()

Outlet_Type
Supermarket Type1    5577
Grocery Store        1083
Supermarket Type3     935
Supermarket Type2     928
Name: count, dtype: int64

In [53]:
df['Outlet_Age'] = 2026 - df['Outlet_Establishment_Year']

In [54]:
df.drop(['Outlet_Establishment_Year'], axis=1, inplace=True)

In [55]:
df = df.drop('Item_Identifier',axis=1)

df

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Item_Type_Combined,Outlet_Age
0,9.300,Low Fat,0.016047,Dairy,249.8092,OUT049,Medium,Tier 1,Supermarket Type1,3735.1380,Food,27
1,5.920,Regular,0.019278,Soft Drinks,48.2692,OUT018,Medium,Tier 3,Supermarket Type2,443.4228,Drinks,17
2,17.500,Low Fat,0.016760,Meat,141.6180,OUT049,Medium,Tier 1,Supermarket Type1,2097.2700,Food,27
3,19.200,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,NaN,Tier 3,Grocery Store,732.3800,Food,28
4,8.930,Low Fat,0.000000,Household,53.8614,OUT013,High,Tier 3,Supermarket Type1,994.7052,Non-Consumable,39
...,...,...,...,...,...,...,...,...,...,...,...,...
8518,6.865,Low Fat,0.056783,Snack Foods,214.5218,OUT013,High,Tier 3,Supermarket Type1,2778.3834,Food,39
8519,8.380,Regular,0.046982,Baking Goods,108.1570,OUT045,NaN,Tier 2,Supermarket Type1,549.2850,Food,24
8520,10.600,Low Fat,0.035186,Health and Hygiene,85.1224,OUT035,Small,Tier 2,Supermarket Type1,1193.1136,Non-Consumable,22
8521,7.210,Regular,0.145221,Snack Foods,103.1332,OUT018,Medium,Tier 3,Supermarket Type2,1845.5976,Food,17


In [56]:
df = df.drop('Outlet_Identifier',axis=1)

df

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Item_Type_Combined,Outlet_Age
0,9.300,Low Fat,0.016047,Dairy,249.8092,Medium,Tier 1,Supermarket Type1,3735.1380,Food,27
1,5.920,Regular,0.019278,Soft Drinks,48.2692,Medium,Tier 3,Supermarket Type2,443.4228,Drinks,17
2,17.500,Low Fat,0.016760,Meat,141.6180,Medium,Tier 1,Supermarket Type1,2097.2700,Food,27
3,19.200,Regular,0.000000,Fruits and Vegetables,182.0950,NaN,Tier 3,Grocery Store,732.3800,Food,28
4,8.930,Low Fat,0.000000,Household,53.8614,High,Tier 3,Supermarket Type1,994.7052,Non-Consumable,39
...,...,...,...,...,...,...,...,...,...,...,...
8518,6.865,Low Fat,0.056783,Snack Foods,214.5218,High,Tier 3,Supermarket Type1,2778.3834,Food,39
8519,8.380,Regular,0.046982,Baking Goods,108.1570,NaN,Tier 2,Supermarket Type1,549.2850,Food,24
8520,10.600,Low Fat,0.035186,Health and Hygiene,85.1224,Small,Tier 2,Supermarket Type1,1193.1136,Non-Consumable,22
8521,7.210,Regular,0.145221,Snack Foods,103.1332,Medium,Tier 3,Supermarket Type2,1845.5976,Food,17


In [57]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Item_Weight           7060 non-null   float64
 1   Item_Fat_Content      8523 non-null   str    
 2   Item_Visibility       8523 non-null   float64
 3   Item_Type             8523 non-null   str    
 4   Item_MRP              8523 non-null   float64
 5   Outlet_Size           6113 non-null   str    
 6   Outlet_Location_Type  8523 non-null   str    
 7   Outlet_Type           8523 non-null   str    
 8   Item_Outlet_Sales     8523 non-null   float64
 9   Item_Type_Combined    8523 non-null   str    
 10  Outlet_Age            8523 non-null   int64  
dtypes: float64(4), int64(1), str(6)
memory usage: 732.6 KB


In [66]:
ct = df.select_dtypes(include='str').columns
ct

Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Type_Combined'],
      dtype='str')

In [71]:
df['Item_Weight'] = df['Item_Weight'].fillna(df['Item_Weight'].mean(), inplace=True)

df['Outlet_Size'] = df['Outlet_Size'].fillna(df['Outlet_Size'].mode()[0], inplace=True)

C:\Users\HARSH GOSALIA\AppData\Local\Temp\ipykernel_13572\4075300334.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Item_Weight'] = df['Item_Weight'].fillna(df['Item_Weight'].mean(), inplace=True)
C:\Users\HARSH GOSALIA\AppData\Local\Temp\ipykernel_13572\4075300334.py:3: ChainedAssignmentError: A value is being set on a copy

In [72]:
df.isnull().sum()

Item_Weight             0
Item_Fat_Content        0
Item_Visibility         0
Item_Type               0
Item_MRP                0
Outlet_Size             0
Outlet_Location_Type    0
Outlet_Type             0
Item_Outlet_Sales       0
Item_Type_Combined      0
Outlet_Age              0
dtype: int64

In [73]:
df = pd.get_dummies(df, columns=ct, drop_first=True).astype(int)
df

,Item_Weight,Item_Visibility,Item_MRP,Item_Outlet_Sales,Outlet_Age,Item_Fat_Content_Regular,Item_Type_Breads,Item_Type_Breakfast,Item_Type_Canned,Item_Type_Dairy,...,Item_Type_Starchy Foods,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Item_Type_Combined_Food,Item_Type_Combined_Non-Consumable
0,9,0,249,3735,27,0,0,0,0,1,...,0,1,0,0,0,1,0,0,1,0
1,5,0,48,443,17,1,0,0,0,0,...,0,1,0,0,1,0,1,0,0,0
2,17,0,141,2097,27,0,0,0,0,0,...,0,1,0,0,0,1,0,0,1,0
3,19,0,182,732,28,1,0,0,0,0,...,0,1,0,0,1,0,0,0,1,0
4,8,0,53,994,39,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,6,0,214,2778,39,0,0,0,0,0,...,0,0,0,0,1,1,0,0,1,0
8519,8,0,108,549,24,1,0,0,0,0,...,0,1,0,1,0,1,0,0,1,0
8520,10,0,85,1193,22,0,0,0,0,0,...,0,0,1,1,0,1,0,0,0,1
8521,7,0,103,1845,17,1,0,0,0,0,...,0,1,0,0,1,0,1,0,1,0


In [74]:
X = df.drop('Item_Outlet_Sales', axis=1)
y = df['Item_Outlet_Sales']

In [75]:
df

,Item_Weight,Item_Visibility,Item_MRP,Item_Outlet_Sales,Outlet_Age,Item_Fat_Content_Regular,Item_Type_Breads,Item_Type_Breakfast,Item_Type_Canned,Item_Type_Dairy,...,Item_Type_Starchy Foods,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Item_Type_Combined_Food,Item_Type_Combined_Non-Consumable
0,9,0,249,3735,27,0,0,0,0,1,...,0,1,0,0,0,1,0,0,1,0
1,5,0,48,443,17,1,0,0,0,0,...,0,1,0,0,1,0,1,0,0,0
2,17,0,141,2097,27,0,0,0,0,0,...,0,1,0,0,0,1,0,0,1,0
3,19,0,182,732,28,1,0,0,0,0,...,0,1,0,0,1,0,0,0,1,0
4,8,0,53,994,39,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,6,0,214,2778,39,0,0,0,0,0,...,0,0,0,0,1,1,0,0,1,0
8519,8,0,108,549,24,1,0,0,0,0,...,0,1,0,1,0,1,0,0,1,0
8520,10,0,85,1193,22,0,0,0,0,0,...,0,0,1,1,0,1,0,0,0,1
8521,7,0,103,1845,17,1,0,0,0,0,...,0,1,0,0,1,0,1,0,1,0


In [76]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [77]:

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()

model.fit(X,y)

importances = model.feature_importances_

In [82]:
from sklearn.model_selection import train_test_split

X = df.drop('Item_Outlet_Sales',axis=1)

y = df['Item_Outlet_Sales']

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42)

In [83]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train,y_train)

pred = model.predict(X_test)

In [84]:
from sklearn.metrics import r2_score,mean_squared_error

print("R2:",r2_score(y_test,pred))
print("RMSE:",mean_squared_error(y_test,pred))

R2: 0.5799984098848798
RMSE: 1141551.559339185


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()

model.fit(X,y)

importances = model.feature_importances_




In [87]:
y_pred = model.predict(X_test)
print("R2:",r2_score(y_test,y_pred))
print("RMSE:",mean_squared_error(y_test,y_pred))

R2: 0.9275710985667359
RMSE: 196859.55319765394
